# 04 Association rules (M2: first set of rules)
Goal (Q2): which combinations of financial, academic and background factors **co-occur with Dropout**?

We use only columns that are known early (admission info and first-semester results), turn them into categories,
and mine rules of the form **IF A and B THEN Target=Dropout**.

How to read a rule:
- **support**: share of all students who have both the IF part and Dropout
- **confidence**: among students with the IF part, the share who dropped out
- **lift**: confidence divided by the overall Dropout rate. Lift above 1 means the IF part is linked with Dropout more than chance

In [ ]:
!pip install -q ucimlrepo mlxtend

## 1. Load data

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd, numpy as np, time

ds = fetch_ucirepo(id=697)
df = pd.concat([ds.data.features, ds.data.targets], axis=1)
print(df.shape)
df["Target"].value_counts()

## 2. Turn columns into categories (discretisation)

In [ ]:
# Association rules need categories, not numbers, so we turn each chosen column into readable labels.
def yn(s):                      # 1/0 -> yes/no
    return s.map({1: "yes", 0: "no"})

items = pd.DataFrame(index=df.index)
items["Debtor"]                   = "Debtor=" + yn(df["Debtor"])
items["Tuition fees up to date"]  = "Tuition up to date=" + yn(df["Tuition fees up to date"])
items["Scholarship holder"]       = "Scholarship=" + yn(df["Scholarship holder"])
items["Displaced"]                = "Displaced=" + yn(df["Displaced"])
items["Gender"]                   = "Gender=" + df["Gender"].map({1: "male", 0: "female"})

items["Age"] = "Age=" + pd.cut(df["Age at enrollment"], bins=[0, 19, 24, 200],
                               labels=["<=19", "20-24", "25+"]).astype(str)
items["Admission grade"] = "Admission grade=" + pd.qcut(df["Admission grade"].rank(method="first"), q=3,
                               labels=["low", "mid", "high"]).astype(str)
items["Prev qualification grade"] = "Prev. qualification grade=" + pd.qcut(df["Previous qualification (grade)"].rank(method="first"), q=3,
                               labels=["low", "mid", "high"]).astype(str)

# first-semester results: many students have 0, so we use fixed bins instead of quantiles
items["1st sem approved"] = "1st sem approved=" + pd.cut(df["Curricular units 1st sem (approved)"],
                               bins=[-1, 0, 3, 5, 100], labels=["none", "1-3", "4-5", "6+"]).astype(str)
items["1st sem grade"] = "1st sem grade=" + pd.cut(df["Curricular units 1st sem (grade)"],
                               bins=[-1, 0, 11, 13, 100], labels=["none", "low", "mid", "high"]).astype(str)

items["Target"] = "Target=" + df["Target"]

onehot = pd.get_dummies(items, prefix="", prefix_sep="", dtype=bool)
print(onehot.shape)
items.head()

In [ ]:
# Baseline: how common is Dropout overall? A rule is only interesting if it beats this.
base = onehot["Target=Dropout"].mean()
print(f"Overall Dropout rate: {base:.3f}")

## 3. Frequent itemsets and rules (Apriori)
Thresholds: min_support = 0.05, min_confidence = 0.50, lift >= 1.2 (justification in section 6).

In [ ]:
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

MIN_SUP, MIN_CONF, MIN_LIFT = 0.05, 0.50, 1.2

t0 = time.perf_counter()
freq = apriori(onehot, min_support=MIN_SUP, use_colnames=True)
t_apriori = time.perf_counter() - t0
print("Frequent itemsets (Apriori):", len(freq), f"| time {t_apriori:.2f}s")

def get_rules(freq_itemsets, min_conf):
    try:
        return association_rules(freq_itemsets, num_itemsets=len(onehot), metric="confidence", min_threshold=min_conf)
    except TypeError:           # older mlxtend versions
        return association_rules(freq_itemsets, metric="confidence", min_threshold=min_conf)

rules = get_rules(freq, MIN_CONF)
print("All rules above confidence", MIN_CONF, ":", len(rules))

## 4. Rules that predict Dropout

In [ ]:
# Keep only rules that predict Dropout, have no Target item on the left side, and have lift above the threshold
is_dropout = rules["consequents"].apply(lambda s: s == frozenset({"Target=Dropout"}))
no_target_left = rules["antecedents"].apply(lambda s: not any(i.startswith("Target=") for i in s))
dropout_rules = rules[is_dropout & no_target_left & (rules["lift"] >= MIN_LIFT)].copy()

dropout_rules["IF (antecedent)"] = dropout_rules["antecedents"].apply(lambda s: "  AND  ".join(sorted(s)))
dropout_rules = dropout_rules.sort_values(["lift", "confidence"], ascending=False)
cols = ["IF (antecedent)", "support", "confidence", "lift"]
print(len(dropout_rules), "Dropout rules with lift >=", MIN_LIFT)
dropout_rules[cols].head(15).round(3).reset_index(drop=True)

## 5. Apriori vs FP-Growth

In [ ]:
# Apriori vs FP-Growth: same frequent itemsets, different speed
t0 = time.perf_counter()
freq_fp = fpgrowth(onehot, min_support=MIN_SUP, use_colnames=True)
t_fp = time.perf_counter() - t0

same = {frozenset(s) for s in freq["itemsets"]} == {frozenset(s) for s in freq_fp["itemsets"]}
print(f"Apriori:   {len(freq)} itemsets, {t_apriori:.3f}s")
print(f"FP-Growth: {len(freq_fp)} itemsets, {t_fp:.3f}s")
print("Same itemsets found:", same)

## 6. Threshold check

In [ ]:
# Why these thresholds? Show how the number of Dropout rules changes with min_support.
rows = []
for sup in [0.03, 0.05, 0.08, 0.10]:
    f = apriori(onehot, min_support=sup, use_colnames=True)
    r = get_rules(f, MIN_CONF)
    m = r["consequents"].apply(lambda s: s == frozenset({"Target=Dropout"})) & (r["lift"] >= MIN_LIFT)
    rows.append({"min_support": sup, "frequent itemsets": len(f), "Dropout rules (conf>=0.5, lift>=1.2)": int(m.sum())})
pd.DataFrame(rows)

## 7. Notes (fill in after reading the output)
- Top 3 rules and what they mean in plain words:
- Which rules look obvious and which look surprising:
- Limits: rules show association, not cause. Students are only labelled for decision support.